In [12]:
from pathlib import Path
import sys

import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

sys.path.insert(0, "..")

from fltabular.task import CATEGORICAL_COLS, ENCODER, SCALER, CostRegressor, evaluator

model_path = Path("/workspaces/flower/fl-tabular/final_model.pt")
test_path = Path("/workspaces/flower/costami-1/test_db.csv")

test_df = pd.read_csv(test_path).dropna().copy()
feature_cols = [col for col in test_df.columns if col not in {"CRF01", "COST_BL"}]

X_test = test_df[feature_cols].copy()
y_test = test_df["COST_BL"].copy()

categorical_cols = [col for col in X_test.columns if col in CATEGORICAL_COLS]
if categorical_cols:
    X_test[categorical_cols] = ENCODER.transform(X_test[categorical_cols])
X_test[X_test.columns] = SCALER.transform(X_test)

X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)
test_loader = DataLoader(
    TensorDataset(X_test_tensor, y_test_tensor),
    batch_size=8,
    shuffle=False,
)

model = CostRegressor(input_dim=X_test_tensor.shape[1])
state_dict = torch.load(model_path, map_location="cpu")
model.load_state_dict(state_dict)

mae, mse, rmse, r2 = evaluator(model, test_loader)
print(f"MAE: {mae:.6f}")
print(f"MSE: {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R2: {r2:.6f}")

MAE: 2445.809326
MSE: 14339499.000000
RMSE: 3786.753174
R2: -0.749503
